In [ ]:
# =====================================================================
# BLOCO 1: Configuração geral (instalação, secrets, clientes, pipeline)
# Rode este bloco UMA VEZ por sessão do Colab.
# =====================================================================
!pip install -q trafilatura google-genai notion-client

import os
import sys
from google import genai
from google.colab import userdata
from notion_client import Client as NotionClient

# --- Secrets -------------------------------------------------------------
try:
    CHAVE_GEMINI = userdata.get('GEMINI_API_KEY')
except Exception:
    CHAVE_GEMINI = None
    print("ERRO: Secret 'GEMINI_API_KEY' não encontrado.")

try:
    NOTION_DATABASE_ID = userdata.get('NOTION_DATABASE_ID_MTG_TRANSLATOR')
except Exception:
    NOTION_DATABASE_ID = None
    print("ERRO: Secret 'NOTION_DATABASE_ID_MTG_TRANSLATOR' não encontrado.")

gemini_client = genai.Client(api_key=CHAVE_GEMINI) if CHAVE_GEMINI else None
notion = NotionClient(auth=userdata.get("NOTION_TOKEN_MTG_TRANSLATOR"))

# --- Configurações fixas ---------------------------------------------------
MODELO_ESCOLHIDO = 'gemini-2.5-flash'  # ou 'gemini-3.1-pro' para priorizar qualidade literária
STATUS_NAME = "Not started"            # coluna "Status" — sempre esse valor na publicação inicial

# --- Clonagem/atualização do repositório (traz os módulos e o glossário) --
GITHUB_REPO_URL = "https://github.com/ThaisMosken/magic_story_translator.git"
GLOSSARY_FILE_PATH = "glossary_mtg.md"  # caminho do .md dentro do repositório
FORCE_UPDATE_GLOSSARY = False             # True para puxar mudanças do repo a cada execução

REPO_LOCAL_FOLDER = "magic_story_translator"

if not os.path.exists(REPO_LOCAL_FOLDER):
    !git clone {GITHUB_REPO_URL}
elif FORCE_UPDATE_GLOSSARY:
    !git -C {REPO_LOCAL_FOLDER} pull

sys.path.append(f'/content/{REPO_LOCAL_FOLDER}')

# --- Módulos do projeto (importados do repositório clonado) ----------------
from src.glossary_manager import load_glossary
from src.pipeline import build_pipeline

MTG_VOCABULARY = load_glossary(os.path.join(REPO_LOCAL_FOLDER, GLOSSARY_FILE_PATH))
print(f"✅ Glossário carregado com {len(MTG_VOCABULARY)} termos.")

# translate_and_publish já sai pronto para usar no BLOCO 2, sem precisar
# repassar clientes/configurações a cada conto.
translate_and_publish = build_pipeline(
    gemini_client=gemini_client,
    notion_client=notion,
    database_id=NOTION_DATABASE_ID,
    model_name=MODELO_ESCOLHIDO,
    vocabulary=MTG_VOCABULARY,
    status_name=STATUS_NAME,
)

print("✅ Ambiente configurado. Preencha o BLOCO 2 com os dados do conto e execute.")

In [ ]:
# =====================================================================
# BLOCO 2: Conto atual — preencha e rode este bloco para cada tradução
# =====================================================================
STORY_URL = "COLE_A_URL_AQUI"
ARC_NAME = "COLE_O_NOME_DO_ARCO_AQUI"       # precisa já existir como opção na coluna "Arc" (select)
TAGS = ["TAG_1", "TAG_2"]             # coluna "Tags" (multi-select)

# Para realizar teste sem fazer chamada à API do Gemini, selecione dry_run=True
translate_and_publish(STORY_URL, ARC_NAME, TAGS, dry_run=False)